# `06_qlora_finetuning.ipynb` — 4-bit NF4 QLoRA (`Q1`, Central CAS Lab, 35–85 min Session)

## 1. How Does 4-bit NF4 QLoRA Work Under the Hood?
**QLoRA** (Dettmers et al., 2023) enables fine-tuning multi-billion-parameter LLMs on consumer or classroom GPUs by combining three systems innovations:
1. **4-bit NormalFloat (`nf4`) Quantization:**
   Pretrained neural network weights follow a zero-centered normal distribution $\mathcal{N}(0, \sigma^2)$. Unlike uniform `int4` quantization (which wastes bins in the tails), `nf4` places $2^4 = 16$ quantization levels at the **equal-probability quantiles** of the standard normal distribution—making it information-theoretically optimal for normally distributed weights.
2. **Double Quantization (`bnb_4bit_use_double_quant=True`):**
   Blockwise quantization stores a 32-bit scaling constant $c_1$ for every block of 64 weights (`0.5 bits/parameter` overhead). Double quantization quantizes those scaling constants themselves into 8-bit floats (`c_2` per 256 blocks), reducing the overhead from `0.5` to **`0.127 bits/parameter`**.
3. **Mixed-Precision Forward/Backward Pass (`bnb_4bit_compute_dtype=torch.bfloat16`):**
   - **Storage dtype:** Base weights $W_0$ sit in GPU VRAM in 4-bit `nf4` (`~1.40 GB` for Gemma-2-2B instead of `~4.99 GB` in `bf16`).
   - **Compute dtype:** During each forward and backward matrix multiplication, the 4-bit block is dequantized on-the-fly into `bfloat16`, multiplied with the `bfloat16` activation $x$, and added to the `bfloat16` LoRA path $\frac{\alpha}{r} B A x$. Gradients flow **only** into the `bfloat16` LoRA matrices $A$ and $B$.

```text
                        Output Activation h = h_base + (α/r) · h_lora
                                       ▲
                                       │
                                      (+)  ◄── Elementwise Sum in bfloat16
                                     ╱   ╲
                                    ╱     ╲
         ┌─────────────────────────┘       └──────────────────────────┐
         │                                                            │
         │  h_base = dequant(W_0) · x        h_lora = (α/r) · B · (A · x)
         │                                                            │
┌────────┴────────────────────────┐          ┌────────────────────────┴────────┐
│   FROZEN BASE WEIGHT W_0        │          │   TRAINABLE LoRA ADAPTER (B·A)  │
│   [d_out × d_in]                │          │                                 │
│                                 │          │   ┌─────────────────────────┐   │
│ • Stored in 4-bit NF4 (~1.4 GB) │          │   │ Matrix B [d_out × r=16] │   │
│ • Dequantized on-the-fly to     │          │   │ (bfloat16, init = 0)    │   │
│   bfloat16 during matmul        │          │   └────────────▲────────────┘   │
│ • requires_grad = False         │          │                │ (16-dim bottleneck)
│ • 0 optimizer states            │          │   ┌────────────┴────────────┐   │
│   (NEVER MODIFIED)              │          │   │ Matrix A [r=16 × d_in]  │   │
└────────▲────────────────────────┘          │   │ (bfloat16, init ~ N)    │   │
         │                                   │   └────────────▲────────────┘   │
         │                                   │                │                │
         │                                   │ • requires_grad = True          │
         │                                   │ • Updated by SFTTrainer AdamW   │
         │                                   └────────────────▲────────────────┘
         │                                                    │
         └─────────────────────────┐       ┌──────────────────┘
                                    ╲     ╱
                                     ╲   ╱
                                       │
                             Input Activation x (bfloat16)
```

> **Crucial Pedagogical Point:** QLoRA is a **VRAM-compression technique**, not a regularizer or accuracy booster over 16-bit LoRA. Success means matching 16-bit LoRA's F1 score within $<0.5$ points while cutting base weight memory by **~72%**.

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from germeval_llm.convert import ensure_derived_datasets_ready
ensure_derived_datasets_ready()
print("Repository initialized:", ROOT.name)


In [ ]:
from germeval_llm.training import estimate_gemma_parameter_and_memory_profile

rows = []
for r in [8, 16, 32]:
    rows.append(estimate_gemma_parameter_and_memory_profile(lora_rank=r, quantization='bf16_16bit'))
    rows.append(estimate_gemma_parameter_and_memory_profile(lora_rank=r, quantization='nf4_4bit'))
df_mem = pd.DataFrame(rows)
display(df_mem[['quantization', 'lora_rank', 'trainable_parameters', 'trainable_percent', 'base_weights_mb', 'adapter_checkpoint_mb', 'estimated_peak_vram_mb']])

In [ ]:
bench = pd.read_csv('results/metrics/benchmark.csv')
display(bench[bench['run_id'].isin(['P3_dev_optimized_10shot', 'L1_lora_r16', 'Q1_qlora_r16'])])

## 2. Live Execution Option (Google Colab L4 GPU or Local MPS)
Set `RUN_LIVE_SMOKE = True` in the cell below (or open [`notebooks/11_colab_l4_live_demo.ipynb`](11_colab_l4_live_demo.ipynb)) to run live QLoRA/LoRA fine-tuning and test-set inference right inside the notebook:

In [ ]:
RUN_LIVE_SMOKE = False  # Set True on Google Colab (L4 GPU) or local MPS to run live training!

if RUN_LIVE_SMOKE:
    from germeval_llm.training import run_sft_adapter_training
    summary = run_sft_adapter_training(
        config_path='configs/qlora.yaml',
        train_jsonl_path='data/derived/task_c/train.jsonl',
        dev_jsonl_path='data/derived/task_c/dev.jsonl',
        output_adapter_dir='models/adapters/live_classroom_qlora',
        subset_size=500,
        max_steps_override=100,
    )
    print(json.dumps(summary, indent=2))
else:
    print('Precomputed smoke training summary (results/runs/smoke_task_c_lora_train_summary.json):')
    print(Path('results/runs/smoke_task_c_lora_train_summary.json').read_text())

### Key Takeaways
- 4-bit NF4 QLoRA reduces base model weight memory by **72%** (from `4,986.5 MB` to `1,402.4 MB`), cutting total training VRAM from `~7.2 GB` to `~2.9 GB`.
- Task C1 and C2 Micro-F1 match 16-bit LoRA within `~0.2–0.4` F1 points (`0.8360` vs. `0.8384` on `test_syn`).